In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
coursera = pd.read_csv("Coursera.csv")
print(coursera.columns)
print(coursera.head(2))

In [ ]:
udemy = pd.read_csv("Udemy.csv")
print(udemy.columns)
print(udemy.head(2))

In [ ]:
edx = pd.read_csv("edx.csv")
print(edx.columns)
print(edx.head(2))

In [ ]:
import pandas as pd
coursera = pd.read_csv("Coursera.csv")
udemy = pd.read_csv("Udemy.csv")
edx = pd.read_csv("edx.csv") #dataset loading

In [ ]:
coursera_df = pd.DataFrame()
coursera_df["title"] = coursera["course"]
coursera_df["skills"] = coursera["skills"]
coursera_df["provider"] = "Coursera"
coursera_df["level"] = coursera["level"]
coursera_df["rating"] = coursera["rating"]
coursera_df["description"] = coursera["skills"]
udemy_df = pd.DataFrame()
udemy_df["title"] = udemy["title"]
udemy_df["skills"] = udemy["description"]
udemy_df["provider"] = "Udemy"
udemy_df["level"] = udemy["level"]
udemy_df["rating"] = udemy["rating"]
udemy_df["description"] = udemy["description"]
edx_df = pd.DataFrame()
edx_df["title"] = edx["title"]
edx_df["skills"] = edx["associatedskills"]
edx_df["provider"] = "edX"
edx_df["level"] = edx["level"]
edx_df["rating"] = 0
edx_df["description"] = edx["subject"]#standardizing datasets

In [ ]:
df = pd.concat(
    [coursera_df, udemy_df, edx_df],
    ignore_index=True
) #merge

In [ ]:
df.fillna("", inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)  #clean missing values

In [ ]:
df["combined"] = (
    df["title"].astype(str) + " " +
    df["skills"].astype(str) + " " +
    df["description"].astype(str) + " " +
    df["level"].astype(str)
)  #combined feature creation

In [ ]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text
df["combined"] = df["combined"].apply(clean_text) #text cleaning

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000
)
course_vectors = tfidf.fit_transform(
    df["combined"]
)
course_vectors.shape  #TF-IDF

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
def recommend_courses(
    skills,
    interests,
    goal,
    top_n=10
):
    query = skills + " " + interests + " " + goal
    query = clean_text(query)
    query_vector = tfidf.transform([query])
    similarities = cosine_similarity(
        query_vector,
        course_vectors
    )[0]
    rating_score = (
        pd.to_numeric(
            df["rating"],
            errors="coerce"
        ).fillna(0) / 5
    )
    final_score = (
        similarities * 0.8 +
        rating_score * 0.2
    )
    top_indices = np.argsort(
        final_score
    )[::-1][:top_n]
    results = df.iloc[top_indices][
        [
            "title",
            "provider",
            "level",
            "rating"
        ]
    ].copy()
    results["match_percent"] = (
        final_score[top_indices] * 100
    ).round(2)
    return results  #recommendation function

In [ ]:
recommend_courses(
    skills="Python SQL",
    interests="Data Science",
    goal="Data Analyst"
) #Testing

In [ ]:
recommend_courses(
    skills="Linux Networking",
    interests="Cybersecurity",
    goal="Security Analyst"
)

In [ ]:
recommend_courses(
    skills="TensorFlow Python",
    interests="Artificial Intelligence",
    goal="AI Engineer"
)